# A04 · 07 · Pathway enrichment & GSEA
Enrichr over DE gene sets + prerank GSEA on ranked LFC. Expect (if hypothesis holds): UPR/unfolded-protein-response up with butyrate; TNFα-via-NFκB & inflammatory hallmarks down in Cd as butyrate rises.
> needs `pip install gseapy` (internet for Enrichr libraries).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))   # a04_common.py next to notebooks
import a04_common as ac
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
print('project root:', ac.PROJ)


project root: /storage/users/job37yv/Projects/DECIDE_A04_CdiffOrganoid_Butyrate


In [2]:
try:
    import gseapy as gp
except ImportError:
    print('pip install gseapy  # then re-run'); raise

In [3]:
def load(name): return pd.read_csv(ac.TAB/f'DE_{name}.csv', index_col=0)
name = 'Butyrate_10v0_Cd'   # change to any contrast
r = load(name)
up = r[(r.padj<0.05)&(r.log2FoldChange>1)]['symbol'].dropna().tolist()
dn = r[(r.padj<0.05)&(r.log2FoldChange<-1)]['symbol'].dropna().tolist()
print(name, '| up', len(up), 'down', len(dn))

Butyrate_10v0_Cd | up 1351 down 836


In [4]:
libs = ['MSigDB_Hallmark_2020','GO_Biological_Process_2023','KEGG_2021_Human']
if up:
    enr = gp.enrichr(gene_list=up, gene_sets=libs, outdir=None)
    display(enr.results.sort_values('Adjusted P-value').head(15)[['Gene_set','Term','Adjusted P-value','Genes']])

,Gene_set,Term,Adjusted P-value,Genes
2664,KEGG_2021_Human,Herpes simplex virus 1 infection,0.002453,ZNF331;ZNF573;ZNF132;ZNF891;ZNF607;ZNF606;ZNF5...
2665,KEGG_2021_Human,Arachidonic acid metabolism,0.091750,CYP2B6;PLA2G4D;PLA2G4E;CYP4F2;ALOX15;CYP4F3;GP...
0,MSigDB_Hallmark_2020,KRAS Signaling Dn,0.372298,PNMT;KMT2D;NPY4R;HTR1D;KRT13;LYPD3;ALOX12B;CCD...
1,MSigDB_Hallmark_2020,Apical Surface,0.999984,SLC34A3;HSPB1;LYPD3;BRCA1
4,MSigDB_Hallmark_2020,Spermatogenesis,0.999984,PCSK1N;HSPA1L;CHRM4;GFI1;ALOX15;NPHP1;TNNI3;MTOR
5,MSigDB_Hallmark_2020,Estrogen Response Late,0.999984,SLC26A2;RAB31;ABCA3;ALDH3B1;GPER1;KRT13;CACNA2...
2,MSigDB_Hallmark_2020,Estrogen Response Early,0.999984,SLC26A2;PEX11A;ABCA3;KRT13;ASB13;CELSR2;TOB1;R...
3,MSigDB_Hallmark_2020,Notch Signaling,0.999984,FZD1;FZD7
24,MSigDB_Hallmark_2020,Fatty Acid Metabolism,0.999984,FASN;TP53INP2;CA4;CEL;ACSS1
25,MSigDB_Hallmark_2020,UV Response Dn,0.999984,MAGI2;PMP22;CACNA1A;HAS2


## Prerank GSEA on the full ranked LFC (Hallmark)

In [5]:
rnk = r.dropna(subset=['symbol']).groupby('symbol')['log2FoldChange'].mean().sort_values(ascending=False)
pre = gp.prerank(rnk=rnk.reset_index(), gene_sets='MSigDB_Hallmark_2020', outdir=None, min_size=5, max_size=500)
pre.res2d.sort_values('NES').head(10)[['Term','NES','FDR q-val']]

,Term,NES,FDR q-val
0,TNF-alpha Signaling via NF-kB,-2.662173,0.0
1,Myc Targets V1,-2.413145,0.0
2,IL-6/JAK/STAT3 Signaling,-2.338733,0.0
3,Unfolded Protein Response,-2.008162,0.0
4,Oxidative Phosphorylation,-1.925017,0.0
5,Allograft Rejection,-1.898761,0.000873
6,Myc Targets V2,-1.889701,0.001496
7,Inflammatory Response,-1.824583,0.003273
8,Interferon Gamma Response,-1.803428,0.002909
9,DNA Repair,-1.788007,0.003665


Repeat for `Cd_vs_Mock_0mM` and `Cd_vs_Mock_10mM` to see the inflammatory hallmark shrink across butyrate.